# **HANDS - ON: INTEGRACIÓN ENTRE LLAMA Y WHATSAPP**

Una vez vista la masterclass ***Integración entre Llama y WhatsApp***, se proporciona el siguiente ***colab*** para que **el alumnado pueda practicar** haciendo uso del lenguaje de programación ***python*** y el modelo de ***llama 3.1*** de Meta.

De igual manera, se proporciona la **solución** de este notebook a través del siguiente [enlace](https://colab.research.google.com/drive/1ks58AJ1dS_aeNw_Q4pnYR05Dokr0Dwi6?usp=drive_link).

## **LÓGICA DE NEGOCIO Y MODELO**

### **CONFIGURACIÓN DEL ENTORNO**

#### **COLAB SECRETS**

Para no exponer tu API key directamente en el código, Colab ofrece un panel de Secrets (ícono de llave en la barra lateral izquierda) donde puedes guardarla de forma segura, añadiendo un nombre asociado a la API key para guardarla dentro de una variable y usarla dentro del notebook.

In [ ]:
# Leer credenciales desde Colab Secrets

from google.colab import userdata

# Configura estos nombres en el panel Colab Secrets. No guardes sus valores en el notebook.
VERIFY_TOKEN = userdata.get("VERIFY_TOKEN")
META_APP_SECRET = userdata.get("META_APP_SECRET")  # App Secret de Meta; firma los POST del webhook.
WHATSAPP_TOKEN = userdata.get("WHATSAPP_ACCESS_TOKEN")
PHONE_ID = userdata.get("PHONE_NUMBER_ID")
WABA_ID = userdata.get("WABA_TOKEN_ID")

print("Credenciales cargadas desde Colab Secrets.")


#### **INSTALAR Y ARRANCAR OLLAMA EN COLAB**

Este módulo corre en Colab, así que Ollama no viene preinstalado en la máquina virtual: se instala y se arranca en segundo plano al inicio de cada sesión.

In [ ]:
# Instalar Ollama (requiere zstd, que Colab no trae por defecto)

!apt-get update -qq
!if ! command -v zstd >/dev/null 2>&1; then apt-get install -y zstd; else echo "zstd ya esta instalado."; fi
!if ! command -v ollama >/dev/null 2>&1; then curl -fsSL https://ollama.com/install.sh | sh; else echo "Ollama ya esta instalado."; fi

In [ ]:
# Iniciar el servidor de Ollama en segundo plano

import subprocess
import time
import urllib.error
import urllib.request
import sys

OLLAMA_URL = "http://localhost:11434"
MODELO_OLLAMA = "llama3.1:8b"


def ollama_disponible():
    try:
        with urllib.request.urlopen(f"{OLLAMA_URL}/api/tags", timeout=2) as respuesta:
            return respuesta.status == 200
    except (urllib.error.URLError, TimeoutError):
        return False


if not ollama_disponible():
    proceso_ollama = subprocess.Popen(["ollama", "serve"])
    limite = time.time() + 60
    while time.time() < limite and not ollama_disponible():
        time.sleep(2)
    if not ollama_disponible():
        raise RuntimeError("Ollama no respondio en http://localhost:11434 dentro de 60 segundos.")
else:
    print("Ya hay un servidor Ollama disponible; se reutilizara.")

# La instalacion repetida es segura y permite importar el cliente en una sesion nueva.
!pip install ollama --quiet
import ollama

# `pull` es idempotente: descarga el modelo si falta y reutiliza el modelo existente.
subprocess.run(["ollama", "pull", MODELO_OLLAMA], check=True)
modelos_ollama = ollama.list()
print(f"Ollama disponible en {OLLAMA_URL}; modelo solicitado: {MODELO_OLLAMA}.")

#### **INSTALAR Y LEVANTAR LLAMA STACK**

In [ ]:
# Instalar dependencias

import importlib.util
import subprocess
import sys

paquetes_llama_stack = ["llama_stack", "llama_stack_client", "mcp"]
if not all(importlib.util.find_spec(paquete) for paquete in paquetes_llama_stack):
    subprocess.run(
        [
            sys.executable, "-m", "pip", "install", "--quiet",
            "llama-stack[starter]", "llama-stack-client", "mcp<2.0",
        ],
        check=True,
    )
else:
    print("Las dependencias de Llama Stack ya estan instaladas.")

In [ ]:
# Instalar Llama Stack (servidor + cliente) y levantarlo con Ollama registrado

import os
import subprocess
import time
import yaml
import llama_stack
from llama_stack_client import LlamaStackClient

os.environ["OLLAMA_URL"] = "http://localhost:11434/v1"
LLAMA_STACK_URL = "http://localhost:8321"

# Mantener la distribucion starter y el formato de configuracion de la solucion del curso.
config_path = os.path.join(
    os.path.dirname(llama_stack.__file__), "distributions", "starter", "config.yaml"
)
if not os.path.isfile(config_path):
    raise FileNotFoundError(
        "No se encontro la configuracion distributions/starter/config.yaml del paquete "
        "llama-stack instalado; se detiene sin cambiar de distribucion o API."
    )

with open(config_path, encoding="utf-8") as f:
    config = yaml.safe_load(f)

modelos_registrados = config["registered_resources"]["models"]
if not any(modelo.get("model_id") == MODELO_OLLAMA for modelo in modelos_registrados):
    modelos_registrados.append({
        "model_id": MODELO_OLLAMA,
        "provider_id": "ollama",
        "provider_model_id": MODELO_OLLAMA,
        "model_type": "llm",
    })

mi_config_path = "/content/mi_config_llama_stack.yaml"
with open(mi_config_path, "w", encoding="utf-8") as f:
    yaml.safe_dump(config, f, default_flow_style=False, sort_keys=False)


def llama_stack_disponible():
    try:
        with urllib.request.urlopen(f"{LLAMA_STACK_URL}/v1/models", timeout=3) as respuesta:
            return respuesta.status == 200
    except (urllib.error.URLError, TimeoutError):
        return False


if not llama_stack_disponible():
    proceso_llama_stack = subprocess.Popen(
        ["llama", "stack", "run", mi_config_path, "--port", "8321"],
        stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True,
    )
    limite = time.time() + 90
    while time.time() < limite and proceso_llama_stack.poll() is None and not llama_stack_disponible():
        time.sleep(2)

if not llama_stack_disponible():
    if "proceso_llama_stack" in globals() and proceso_llama_stack.poll() is not None:
        salida = proceso_llama_stack.stdout.read() if proceso_llama_stack.stdout else ""
        raise RuntimeError(f"Llama Stack termino antes de estar disponible. Salida:\n{salida}")
    raise RuntimeError("Llama Stack no respondio en http://localhost:8321/v1/models dentro de 90 segundos.")

# Conservar el cliente e interfaz de la solucion oficial.
client = LlamaStackClient(base_url=LLAMA_STACK_URL, timeout=300)
if not hasattr(client, "conversations") or not hasattr(client, "responses"):
    raise RuntimeError(
        "El cliente instalado no expone conversations y responses como requiere "
        "la solucion oficial del curso; no se aplicara otra API."
    )

print("Llama Stack disponible en", LLAMA_STACK_URL)
print("Modelo registrado: ollama/" + MODELO_OLLAMA)
print("Cliente LlamaStackClient inicializado; conversaciones y responses disponibles.")

### **CONSTRUCCIÓN DE LA LÓGICA DEL ASISTENTE**

In [ ]:
# Configurar el servidor del webhook

from fastapi import FastAPI, Request, BackgroundTasks
import requests
import json

app = FastAPI()

mensajes_recibidos = []


In [ ]:
# Definir función de normalización de número

def normalizar_numero_mx(numero):
    if numero.startswith("521") and len(numero) == 13:
        return "52" + numero[3:]
    return numero

#### **CONSTRUIR LA MEMORIA DE CONVERSACIÓN**

Se define una estructura en memoria que asocia cada número de teléfono con su `conversation_id` de **Llama Stack** — sin esto, Llama trataría cada mensaje como si fuera el primero de la conversación.

In [ ]:
# Una conversacion de Llama Stack por numero de telefono

conversaciones_por_numero = {}

def obtener_conversacion(numero):
    if numero not in conversaciones_por_numero:
        conversacion = client.conversations.create()
        conversaciones_por_numero[numero] = conversacion.id
    return conversaciones_por_numero[numero]

#### **DEFINIR LA HERRAMIENTA: CONSULTAR DISPONIBILIDAD Y PREGUNTAS FRECUENTES**

Hasta el Tema anterior, Llama solo generaba texto a partir de texto — si un usuario preguntaba por un horario, el modelo podía inventar una respuesta que sonara razonable, sin ninguna garantía de que fuera real. **Function calling** resuelve esto: se le describe a Llama una función real de tu código (nombre, qué hace, qué parámetros necesita), y el modelo, cuando lo considera necesario, pide que se ejecute esa función en vez de responder directamente.

Aquí se definen dos herramientas: `consultar_disponibilidad`, que revisa horarios reales del taller, y `buscar_pregunta_frecuente`, que busca la respuesta a dudas comunes (horarios, garantía, formas de pago) en una base de preguntas frecuentes real — una alternativa más ligera a montar un pipeline de RAG completo (embeddings + recuperación) aparte: en vez de eso, la búsqueda se expone como una función más que Llama puede pedir ejecutar, bajo el mismo mecanismo de function calling.

In [ ]:
# Definir la funcion real y su descripcion para Llama: consultar disponibilidad

import re
from datetime import datetime

DISPONIBILIDAD = {
    "2026-10-06": ["09:00", "11:00", "15:00"],
    "2026-10-07": ["10:00", "13:00"],
}

MESES = {
    "enero": 1, "febrero": 2, "marzo": 3, "abril": 4, "mayo": 5, "junio": 6,
    "julio": 7, "agosto": 8, "septiembre": 9, "setiembre": 9, "octubre": 10,
    "noviembre": 11, "diciembre": 12,
}

def normalizar_fecha(fecha_texto):
    fecha_texto = fecha_texto.strip().lower()

    if re.fullmatch(r"\d{4}-\d{2}-\d{2}", fecha_texto):
        _, mes, dia = fecha_texto.split("-")
        return f"{datetime.now().year:04d}-{mes}-{dia}"

    match = re.search(r"(\d{1,2})\s+de\s+([a-zñ]+)(?:\s+de\s+(\d{4}))?", fecha_texto)
    if match:
        dia, mes_texto, anio_texto = match.groups()
        mes = MESES.get(mes_texto)
        if mes:
            anio = int(anio_texto) if anio_texto else datetime.now().year
            return f"{anio:04d}-{mes:02d}-{int(dia):02d}"

    return fecha_texto

def consultar_disponibilidad(fecha=None):
    if not fecha:
        dias = ", ".join(DISPONIBILIDAD.keys())
        return f"Los dias con horarios disponibles son: {dias}. Dime cual te interesa para darte los horarios exactos."

    fecha = normalizar_fecha(fecha)
    horarios = DISPONIBILIDAD.get(fecha, [])
    if not horarios:
        return f"No hay horarios disponibles para el {fecha}."
    return f"Horarios disponibles para el {fecha}: {', '.join(horarios)}."

HERRAMIENTA_DISPONIBILIDAD = {
    "type": "function",
    "name": "consultar_disponibilidad",
    "description": "Consulta los horarios disponibles para llevar el auto al taller. Si el usuario da una fecha especifica, se le devuelven los horarios de ese dia; si pregunta de forma general (sin fecha), se le devuelve la lista de dias con disponibilidad.",
    "parameters": {
        "type": "object",
        "properties": {
            "fecha": {
                "type": "string",
                "description": "Fecha a consultar, en formato YYYY-MM-DD. Si el usuario no menciona el año, asume que es el año actual. Omite este parametro si el usuario no dio una fecha especifica."
            }
        },
        "required": []
    }
}

In [ ]:
# Definir la funcion real y su descripcion para Llama: buscar pregunta frecuente

import unicodedata

def quitar_acentos(texto):
    return ''.join(c for c in unicodedata.normalize('NFD', texto) if unicodedata.category(c) != 'Mn')

PREGUNTAS_FRECUENTES = {
    "horario": "El taller abre de lunes a sabado, de 8:00 a 18:00.",
    "garantia": "Todas las reparaciones tienen 90 dias de garantia en mano de obra.",   # raiz, no palabra completa
    "pago": "Aceptamos efectivo, tarjeta y transferencia.",
    "cotizacion": "La cotizacion es gratuita, solo se cobra si decides hacer la reparacion.",  # raiz
}

def buscar_pregunta_frecuente(pregunta):
    pregunta_norm = quitar_acentos(pregunta.lower())
    for raiz, respuesta in PREGUNTAS_FRECUENTES.items():
        if raiz in pregunta_norm:
            return respuesta
    return "No tenemos una respuesta guardada para esa pregunta; un miembro del equipo puede confirmarte directamente."

HERRAMIENTA_FAQ = {
    "type": "function",
    "name": "buscar_pregunta_frecuente",
    "description": "Busca la respuesta a una pregunta frecuente del taller (horarios, garantia, formas de pago, cotizaciones) en una base de preguntas frecuentes real.",
    "parameters": {
        "type": "object",
        "properties": {
            "pregunta": {
                "type": "string",
                "description": "La pregunta del usuario, en texto libre."
            }
        },
        "required": ["pregunta"]
    }
}

HERRAMIENTAS = [HERRAMIENTA_DISPONIBILIDAD, HERRAMIENTA_FAQ]
FUNCIONES_DISPONIBLES = {
    "consultar_disponibilidad": consultar_disponibilidad,
    "buscar_pregunta_frecuente": buscar_pregunta_frecuente,
}

#### **GENERAR LA RESPUESTA CON HISTORIAL Y FUNCTION CALLING**

Se reenvía el historial reciente a Llama junto con las dos herramientas disponibles. Si el modelo pide ejecutar alguna (`respuesta.output` trae uno o más elementos con `type == "function_call"`), el código la ejecuta de verdad con los argumentos que Llama propuso, le devuelve el resultado en una segunda llamada, y solo entonces se genera la respuesta final en lenguaje natural. Las reglas fijas (urgencia y escalamiento) se resuelven antes de llegar aquí — ver la sección de endpoints.

In [ ]:
# Definir la funcion de generacion de respuesta con memoria y function calling

SYSTEM_PROMPT = (
    "Eres el asistente de un taller mecanico. "
    "Responde de forma breve (2-3 lineas). "
    "Solo usa una herramienta cuando el usuario pregunte algo especifico que la herramienta pueda responder "
    "(disponibilidad de una fecha, o una pregunta frecuente concreta). No uses ninguna herramienta para "
    "confirmaciones o respuestas cortas del usuario (como 'si, por favor' o una hora). "
    "No puedes agendar, confirmar ni programar citas de ninguna forma — solo puedes consultar disponibilidad. "
    "Si el usuario quiere agendar una cita, dile que un miembro del equipo se pondra en contacto para confirmarla. "
    "Si una herramienta no encuentra una respuesta, dilo honestamente en vez de inventar un dato. "
    "Si una herramienta SI regresa un resultado real, siempre comunicalo claramente al usuario; nunca lo ignores ni te niegues a compartirlo. "
    "Responde siempre de forma directa con la informacion final, basandote unicamente en el resultado real de las "
    "herramientas; nunca inventes datos que no vengan de una herramienta. "
    "Nunca describas tu proceso, ni menciones nombres de funciones, parametros o claves de un resultado."
)

def generar_respuesta(numero, texto_usuario):
    conversation_id = obtener_conversacion(numero)
    respuesta = client.responses.create(
        model="ollama/llama3.1:8b",
        input=texto_usuario,
        conversation=conversation_id,
        instructions=SYSTEM_PROMPT,
        tools=HERRAMIENTAS,
    )

    llamadas_funcion = [item for item in respuesta.output if item.type == "function_call"]
    if llamadas_funcion:
        resultados = []
        for llamada in llamadas_funcion:
            funcion = FUNCIONES_DISPONIBLES[llamada.name]
            argumentos = json.loads(llamada.arguments)
            resultado = funcion(**argumentos)
            resultados.append({
                "type": "function_call_output",
                "call_id": llamada.call_id,
                "output": resultado,
            })
        respuesta = client.responses.create(
            model="ollama/llama3.1:8b",
            input=resultados,
            conversation=conversation_id,
            instructions=SYSTEM_PROMPT,
            tools=HERRAMIENTAS,
        )

    return respuesta.output_text

## **DESPLIEGUE DEL SERVIDOR**

### **DEFINIR LOS ENDPOINTS DEL WEBHOOK**

Se definen los dos endpoints que WhatsApp necesita: uno para la verificación inicial del webhook, y otro para recibir cada mensaje entrante. La recepción delega el trabajo pesado a `BackgroundTasks`, para responderle a Meta de inmediato sin esperar a que Llama genere la respuesta — evitando así que Meta reintente la entrega y se dupliquen las respuestas.

In [ ]:
# Definir endpoint de verificacion de credenciales

from fastapi.responses import PlainTextResponse, Response
import hmac

@app.get("/webhook")
async def verificar_webhook(request: Request):
    params = request.query_params
    token_recibido = params.get("hub.verify_token")
    challenge = params.get("hub.challenge")
    if (
        params.get("hub.mode") == "subscribe"
        and isinstance(VERIFY_TOKEN, str)
        and isinstance(token_recibido, str)
        and hmac.compare_digest(token_recibido, VERIFY_TOKEN)
        and challenge is not None
    ):
        return PlainTextResponse(content=challenge, status_code=200)
    return Response(status_code=403)


In [ ]:
# Definir endpoint de recepcion de mensajes con firma oficial de Meta

import hashlib
import hmac
import json
import re
from fastapi.responses import Response

@app.post("/webhook")
async def recibir_mensaje(request: Request, background_tasks: BackgroundTasks):
    # Validar los bytes exactos recibidos antes de parsear, registrar o programar trabajo.
    raw_body = await request.body()
    firma_recibida = request.headers.get("X-Hub-Signature-256")
    if not isinstance(META_APP_SECRET, str) or not META_APP_SECRET or not isinstance(firma_recibida, str):
        return Response(status_code=401)
    # Limitar la entrada a ASCII canonico: sha256= seguido de 64 digitos hexadecimales.
    if re.fullmatch(r"sha256=[0-9a-fA-F]{64}", firma_recibida) is None:
        return Response(status_code=401)
    firma_esperada = "sha256=" + hmac.new(
        META_APP_SECRET.encode("utf-8"), raw_body, hashlib.sha256
    ).hexdigest()
    if not hmac.compare_digest(firma_recibida, firma_esperada):
        return Response(status_code=401)

    try:
        body = json.loads(raw_body)
    except (json.JSONDecodeError, UnicodeDecodeError):
        return {"status": "ignored"}

    mensajes_recibidos.append(body)
    try:
        mensaje = body["entry"][0]["changes"][0]["value"]["messages"][0]
        numero = normalizar_numero_mx(mensaje["from"])
        texto = mensaje["text"]["body"]
        background_tasks.add_task(procesar_y_responder, numero, texto)
    except (KeyError, IndexError, TypeError):
        pass
    return {"status": "ok"}


In [ ]:
# Definir funcion de envio de respuesta

respuestas_enviadas = []

def enviar_respuesta(numero, texto):
    url = f"https://graph.facebook.com/v25.0/{PHONE_ID}/messages"
    headers = {"Authorization": f"Bearer {WHATSAPP_TOKEN}", "Content-Type": "application/json"}
    payload = {
        "messaging_product": "whatsapp",
        "to": numero,
        "type": "text",
        "text": {"body": texto}
    }

    r = requests.post(url, headers=headers, json=payload)
    respuestas_enviadas.append({"numero": numero, "status": r.status_code, "texto": r.text})

In [ ]:
# Un lock por numero de telefono, para serializar mensajes del mismo usuario

import threading

locks_por_numero = {}  # { numero: threading.Lock() }

def obtener_lock(numero):
    if numero not in locks_por_numero:
        locks_por_numero[numero] = threading.Lock()
    return locks_por_numero[numero]

In [ ]:
# Definir funcion de procesamiento: reglas fijas de urgencia, escalamiento y saludo, o respuesta de Llama

from llama_stack_client import APITimeoutError, APIConnectionError, APIStatusError

FRASES_URGENCIA = ["me quede varado", "no arranca", "choque", "tuve un accidente", "emergencia"]
FRASES_ESCALAMIENTO = ["hablar con alguien", "hablar con una persona", "asistente humano"]
FRASES_SALUDO = ["hola", "buenas", "buenos dias", "buenos d\u00edas", "buenas tardes", "buenas noches", "que tal", "buen dia", "buen d\u00eda"]

def es_solo_saludo(texto_low):
    texto_limpio = texto_low.strip(" ,.!\u00a1\u00bf?")
    return texto_limpio in FRASES_SALUDO

def enviar_aviso_de_error(numero):
    try:
        enviar_respuesta(numero, "Estamos teniendo un problema tecnico. Intenta de nuevo en un momento.")
    except Exception:
        pass

def procesar_y_responder(numero, texto):
    with obtener_lock(numero):
        try:
            texto_low = texto.lower()
            if any(frase in texto_low for frase in FRASES_URGENCIA):
                enviar_respuesta(numero, "Eso suena urgente: comunicate directamente a nuestra linea de asistencia en carretera.")
            elif any(frase in texto_low for frase in FRASES_ESCALAMIENTO):
                enviar_respuesta(numero, "Claro, en un momento un miembro del equipo te contacta directamente.")
            elif es_solo_saludo(texto_low):
                enviar_respuesta(numero, "Hola, soy el asistente del taller mecanico. \u00bfEn que puedo ayudarte?")
            else:
                respuesta = generar_respuesta(numero, texto)
                enviar_respuesta(numero, respuesta)
        except (APITimeoutError, APIConnectionError, APIStatusError) as e:
            print("Error de Llama Stack:", repr(e))
            enviar_aviso_de_error(numero)
        except requests.exceptions.RequestException as e:
            # Si WhatsApp no esta disponible, intentar enviar otro mensaje tambien fallaria.
            print("Error de envio a WhatsApp:", repr(e))
        except Exception as e:
            print("Error de procesamiento:", repr(e))
            enviar_aviso_de_error(numero)

### **EXPONER EL SERVIDOR CON NGROK**

Se abre un túnel público con ngrok y se levanta el servidor en un hilo aparte, para que Meta pueda entregarle los mensajes reales al webhook. La URL impresa (+ `/webhook`) es la que se configura como *Callback URL* en Meta for Developers (WhatsApp > Configuración de la API > Configurar webhooks).

In [ ]:
# Instalar e iniciar tunel publico con ngrok y el servidor del Hands-On

!pip install pyngrok --quiet

from pyngrok import ngrok
import uvicorn
import threading
import time

ngrok.set_auth_token(userdata.get("NGROK_AUTHTOKEN"))

# Mantener una referencia al servidor para poder detenerlo antes del Challenge.
if "hilo" not in globals() or hilo is None or not hilo.is_alive():
    servidor_uvicorn = uvicorn.Server(
        uvicorn.Config(app, host="0.0.0.0", port=8000, log_level="warning")
    )
    hilo = threading.Thread(
        target=servidor_uvicorn.run,
        name="uvicorn-hands-on",
        daemon=True,
    )
    hilo.start()

# Esperar al startup real para no publicar un puerto que aun no este escuchando.
limite_inicio_servidor = time.time() + 30
while hilo.is_alive() and not servidor_uvicorn.started and time.time() < limite_inicio_servidor:
    time.sleep(0.1)
if not hilo.is_alive() or not servidor_uvicorn.started:
    raise RuntimeError("Uvicorn del Hands-On no inicio en el puerto 8000 dentro de 30 segundos.")

if "tunel_publico" not in globals() or tunel_publico is None:
    tunel_publico = ngrok.connect(8000)
print("URL a configurar como Callback URL en el dashboard de Meta:")
print(tunel_publico.public_url + "/webhook")


### **PROBAR EL FLUJO COMPLETO**

Se manda un mensaje preguntando por horarios disponibles en una fecha específica, para confirmar que la respuesta viene de `consultar_disponibilidad` y no de una invención de Llama; y un segundo mensaje pidiendo hablar con una persona, para confirmar que la regla fija de escalamiento sigue funcionando.

In [ ]:
# Escenario 1: Mandar mensajes al bot por disponibilidad general

# Mensaje 1:
# Buenas tardes.

# Mensaje 2:
# Necesito llevar mi carro al taller, ¿qué días tienen espacio?

# Mensaje 3:
# Perfecto, ¿y a qué hora el 6 de octubre?

In [ ]:
# Escenario 2: Mandar mensajes al bot por tema de urgencia

# Mensaje 1:
# Hola. Necesito ayuda.
# Acabo de chocar y mi carro no arranca.

# Mensaje 2:
# Comunícame con un asistente humano.

In [ ]:
# Escenario 3: Mandar mensajes al bot por pago y cotización

# Mensaje 1:
# Buenas, ¿cuánto cuesta una cotización?

# Mensaje 2:
# ¿Y qué métodos de pago aceptan?

# Mensaje 3:
# Perfecto, entonces agéndame para el 7 de octubre a las 10.

In [ ]:
# Escenario 4: Mandar mensajes al bot por temas fuera de alcance

# Mensaje 1:
# Hola, ¿tienen servicio de grúa?

# Mensaje 2:
# ¿También atienden motos, o solo carros?

In [ ]:
# Visualizar resultados

print(f'Conversaciones: {conversaciones_por_numero}\n')
print(f'Mensajes recibidos: {mensajes_recibidos}\n')
print(f'Respuestas enviadas: {respuestas_enviadas}\n')

In [ ]:
# Verificar almacenamiento en Llama Stack

numero_a_revisar = '525626131742'

if numero_a_revisar not in conversaciones_por_numero:
    print("No hay mensajes almacenados en estos momentos.")
else:
    conv_id = conversaciones_por_numero[numero_a_revisar]
    items = client.conversations.items.list(conversation_id=conv_id)

    for item in items.data:
        if item.type == "message":
            print(item.role, "->", item.content)
        elif item.type == "function_call":
            print("function_call ->", item.name, item.arguments)
        elif item.type == "function_call_output":
            print("function_call_output ->", item.output)
        else:
            print(item.type, "->", item)

### **DIAGNÓSTICO Y REINICIO**

Si el bot responde con datos que no corresponden al taller (fechas, textos u otro negocio), antes de reiniciar todo el entorno revisa primero si simplemente falta volver a correr alguna celda de arriba.

In [ ]:
# Reiniciar el almacenamiento en Llama Stack

proceso_llama_stack.terminate()
import time; time.sleep(3)

!rm -rf ~/.llama/distributions/starter

proceso_llama_stack = subprocess.Popen(
    ["llama", "stack", "run", mi_config_path, "--port", "8321"],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True
)
time.sleep(20)  # tarda mas en levantar que Ollama

print("Proceso vivo:", proceso_llama_stack.poll() is None)
if proceso_llama_stack.poll() is not None:
    print("--- Salida del proceso ---")
    print(proceso_llama_stack.stdout.read())
else:
    from llama_stack_client import LlamaStackClient
    client = LlamaStackClient(base_url="http://localhost:8321", timeout=300)
    print("Llama Stack listo.")

In [ ]:
# Limpiar los registros en memoria, para una prueba nueva sin ruido de mensajes anteriores

mensajes_recibidos = []
respuestas_enviadas = []
conversaciones_por_numero = {}

In [ ]:
# Finalizar el servidor del Hands-On y el tunel de ngrok antes del Challenge

servidor_actual = globals().get("servidor_uvicorn")
hilo_actual = globals().get("hilo")
if servidor_actual is not None and hilo_actual is not None and hilo_actual.is_alive():
    servidor_actual.should_exit = True
    hilo_actual.join(timeout=10)
    if hilo_actual.is_alive():
        raise RuntimeError("Uvicorn del Hands-On sigue activo tras esperar 10 segundos; no inicies aun el Challenge.")
    print("Servidor Uvicorn del Hands-On detenido; puerto 8000 liberado.")
elif servidor_actual is None or hilo_actual is None:
    print("No hay un servidor Uvicorn del Hands-On activo que detener.")
else:
    print("El servidor Uvicorn del Hands-On ya estaba detenido.")

# Cerrar el agente/tunel solo cuando esta sesion abrio el tunel del Hands-On.
if globals().get("tunel_publico") is not None:
    ngrok.kill()
    tunel_publico = None
    print("Proceso de ngrok del Hands-On cerrado.")
else:
    print("No hay un tunel ngrok del Hands-On que cerrar.")

# Dejar referencias limpias para que la celda se pueda ejecutar otra vez.
servidor_uvicorn = None
hilo = None


## **CHALLENGE: AGENTE DE ATENCIÓN A PEDIDOS**

Una vez visto el ***Hands-On: Integración Llama + WhatsApp***, se presenta el siguiente reto para que el alumnado pueda repasar y reforzar lo aprendido dentro de la clase.

Se construirá un agente similar, pero para la atención a pedidos de una tienda en línea: la misma memoria de conversación en Llama Stack, las mismas reglas fijas (urgencia, escalamiento y saludo) y las mismas dos herramientas de function calling, ahora como `consultar_pedido` y `buscar_pregunta_frecuente`.

**IMPORTANTE:** Para su revisión, es indispensable que los apartados anteriores se encuentren llenados con el código visto durante la sesión.

### **INSTRUCCIONES:**

**1. Prepara el entorno:** Instala y arranca Ollama en Colab, descarga el modelo Llama 3.1 8B, instala y levanta **Llama Stack** con Ollama registrado, y carga las credenciales de WhatsApp desde **Colab Secrets**.

In [ ]:
# Instalar Ollama (requiere zstd, que Colab no trae por defecto)

# Reutiliza el mismo instalador idempotente de la seccion Hands-On.
!if ! command -v zstd >/dev/null 2>&1; then apt-get update -qq && apt-get install -y zstd; else echo "zstd ya esta instalado."; fi
!if ! command -v ollama >/dev/null 2>&1; then curl -fsSL https://ollama.com/install.sh | sh; else echo "Ollama ya esta instalado."; fi

In [ ]:
# Iniciar el servidor de Ollama en segundo plano

# Usa el mismo servidor/modelo configurados en la seccion Hands-On.
import subprocess
import time
import urllib.error
import urllib.request

OLLAMA_URL = "http://localhost:11434"
MODELO_OLLAMA = "llama3.1:8b"

if "ollama_disponible" not in globals():
    def ollama_disponible():
        try:
            with urllib.request.urlopen(f"{OLLAMA_URL}/api/tags", timeout=2) as respuesta:
                return respuesta.status == 200
        except (urllib.error.URLError, TimeoutError):
            return False

if not ollama_disponible():
    proceso_ollama = subprocess.Popen(["ollama", "serve"])
    limite = time.time() + 60
    while time.time() < limite and not ollama_disponible():
        time.sleep(2)
    if not ollama_disponible():
        raise RuntimeError("Ollama no respondio en http://localhost:11434 dentro de 60 segundos.")
else:
    print("Ya hay un servidor Ollama disponible; se reutilizara.")

!pip install ollama --quiet
import ollama
subprocess.run(["ollama", "pull", MODELO_OLLAMA], check=True)
print(f"Ollama disponible; modelo solicitado: {MODELO_OLLAMA}.")

In [ ]:
# Instalar dependencias

# Mismas dependencias que en el Hands-On; no se fija una version no indicada por el curso.
import importlib.util
import subprocess
import sys

paquetes_llama_stack = ["llama_stack", "llama_stack_client", "mcp"]
if not all(importlib.util.find_spec(paquete) for paquete in paquetes_llama_stack):
    subprocess.run(
        [
            sys.executable, "-m", "pip", "install", "--quiet",
            "llama-stack[starter]", "llama-stack-client", "mcp<2.0",
        ],
        check=True,
    )
else:
    print("Las dependencias de Llama Stack ya estan instaladas.")

In [ ]:
# Instalar Llama Stack (servidor + cliente) y levantarlo con Ollama registrado

# Repite exactamente la configuracion starter/ollama de la seccion Hands-On.
import os
import subprocess
import time
import urllib.error
import urllib.request
import yaml
import llama_stack
from llama_stack_client import LlamaStackClient

os.environ["OLLAMA_URL"] = "http://localhost:11434/v1"
LLAMA_STACK_URL = "http://localhost:8321"
config_path = os.path.join(
    os.path.dirname(llama_stack.__file__), "distributions", "starter", "config.yaml"
)
if not os.path.isfile(config_path):
    raise FileNotFoundError(
        "No se encontro la configuracion distributions/starter/config.yaml del paquete "
        "llama-stack instalado; se detiene sin cambiar de distribucion o API."
    )

with open(config_path, encoding="utf-8") as f:
    config = yaml.safe_load(f)
modelos_registrados = config["registered_resources"]["models"]
if not any(modelo.get("model_id") == MODELO_OLLAMA for modelo in modelos_registrados):
    modelos_registrados.append({
        "model_id": MODELO_OLLAMA,
        "provider_id": "ollama",
        "provider_model_id": MODELO_OLLAMA,
        "model_type": "llm",
    })

mi_config_path = "/content/mi_config_llama_stack.yaml"
with open(mi_config_path, "w", encoding="utf-8") as f:
    yaml.safe_dump(config, f, default_flow_style=False, sort_keys=False)


def llama_stack_disponible():
    try:
        with urllib.request.urlopen(f"{LLAMA_STACK_URL}/v1/models", timeout=3) as respuesta:
            return respuesta.status == 200
    except (urllib.error.URLError, TimeoutError):
        return False


if not llama_stack_disponible():
    proceso_llama_stack = subprocess.Popen(
        ["llama", "stack", "run", mi_config_path, "--port", "8321"],
        stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True,
    )
    limite = time.time() + 90
    while time.time() < limite and proceso_llama_stack.poll() is None and not llama_stack_disponible():
        time.sleep(2)

if not llama_stack_disponible():
    if "proceso_llama_stack" in globals() and proceso_llama_stack.poll() is not None:
        salida = proceso_llama_stack.stdout.read() if proceso_llama_stack.stdout else ""
        raise RuntimeError(f"Llama Stack termino antes de estar disponible. Salida:\n{salida}")
    raise RuntimeError("Llama Stack no respondio en http://localhost:8321/v1/models dentro de 90 segundos.")

client = LlamaStackClient(base_url=LLAMA_STACK_URL, timeout=300)
if not hasattr(client, "conversations") or not hasattr(client, "responses"):
    raise RuntimeError(
        "El cliente instalado no expone conversations y responses como requiere "
        "la solucion oficial del curso; no se aplicara otra API."
    )
print("Llama Stack disponible en", LLAMA_STACK_URL)
print("Modelo registrado: ollama/" + MODELO_OLLAMA)
print("Cliente LlamaStackClient inicializado; conversaciones y responses disponibles.")

In [ ]:
# Leer credenciales desde Colab Secrets

# Usa los mismos secrets que el Hands-On; META_APP_SECRET es el App Secret de Meta,
# necesario para validar X-Hub-Signature-256 en los POST del webhook.
from google.colab import userdata

VERIFY_TOKEN = userdata.get("VERIFY_TOKEN")
META_APP_SECRET = userdata.get("META_APP_SECRET")
WHATSAPP_TOKEN = userdata.get("WHATSAPP_ACCESS_TOKEN")
PHONE_ID = userdata.get("PHONE_NUMBER_ID")
WABA_ID = userdata.get("WABA_TOKEN_ID")
print("Credenciales cargadas desde Colab Secrets.")


**2. Construye la memoria de conversación y las herramientas de function calling:**

   * Igual que en el Hands-On, crea una conversación de **Llama Stack** por número de teléfono (`client.conversations.create()`).
   * Define `consultar_pedido(numero_pedido)`, una función real que consulta el estado de un pedido, junto con su descripción para que Llama la pueda pedir como herramienta. Si el usuario no da un número de pedido, la función debe pedírselo en vez de inventar uno.
   * Define `buscar_pregunta_frecuente(pregunta)`, que busca la respuesta a dudas comunes (envíos, devoluciones, formas de pago, facturas) en una base de preguntas frecuentes real.

In [ ]:
# Conversacion de Llama Stack por numero de telefono (Challenge)

# El mapa del Challenge se mantiene separado del mapa del Hands-On.
conversaciones_por_numero_pedidos = {}

def obtener_conversacion_pedidos(numero):
    if numero not in conversaciones_por_numero_pedidos:
        conversacion = client.conversations.create()
        conversaciones_por_numero_pedidos[numero] = conversacion.id
    return conversaciones_por_numero_pedidos[numero]

In [ ]:
# Definir la funcion real y su descripcion para Llama: consultar pedido

# Dataset local de demostracion del Challenge. Solo estos registros son consultables.
PEDIDOS = {
    "PED-1001": {
        "numero_pedido": "PED-1001",
        "estado": "En preparaci\u00f3n",
        "fecha_pedido": "2026-10-01",
        "productos": ["Cafetera de goteo", "Filtro reutilizable"],
        "entrega_estimada": "2026-10-09",
    },
    "PED-1002": {
        "numero_pedido": "PED-1002",
        "estado": "Enviado",
        "fecha_pedido": "2026-09-29",
        "productos": ["Audifonos inalambricos"],
        "transportista": "Estafeta",
        "guia": "EST123456789MX",
        "entrega_estimada": "2026-10-07",
    },
    "PED-1003": {
        "numero_pedido": "PED-1003",
        "estado": "Entregado",
        "fecha_pedido": "2026-09-24",
        "productos": ["Mochila urbana"],
        "fecha_entrega": "2026-09-28",
    },
    "PED-1004": {
        "numero_pedido": "PED-1004",
        "estado": "En transito",
        "fecha_pedido": "2026-10-02",
        "productos": ["Teclado mecanico"],
        "transportista": "DHL",
        "guia": "DHL987654321MX",
        "entrega_estimada": "2026-10-08",
    },
}

def consultar_pedido(numero_pedido=None):
    if numero_pedido is None or not str(numero_pedido).strip():
        return "Para consultar el estado de tu pedido, comparteme el numero de pedido."

    identificador = str(numero_pedido).strip().upper()
    if identificador.isdigit():
        identificador = f"PED-{identificador.zfill(4)}"

    pedido = PEDIDOS.get(identificador)
    if pedido is None:
        return f"No encontre el pedido {identificador} en los registros disponibles. Verifica el numero de pedido."

    # La respuesta se forma unicamente con campos presentes en el registro local.
    detalles = [
        f"{campo.replace('_', ' ')}: {valor}"
        for campo, valor in pedido.items()
    ]
    return "; ".join(detalles)

HERRAMIENTA_CONSULTAR_PEDIDO = {
    "type": "function",
    "name": "consultar_pedido",
    "description": "Consulta el registro local de un pedido usando el numero proporcionado por el cliente. Si no proporciona numero, no inventes uno: solicita el dato al cliente; el argumento numero_pedido es opcional.",
    "parameters": {
        "type": "object",
        "properties": {
            "numero_pedido": {
                "type": "string",
                "description": "Identificador de pedido, por ejemplo PED-1001. No lo inventes ni lo deduzcas si el cliente no lo dio."
            }
        },
        "required": []
    }
}

In [ ]:
# Definir la funcion real y su descripcion para Llama: buscar pregunta frecuente

import unicodedata

def quitar_acentos_pedidos(texto):
    return ''.join(
        caracter
        for caracter in unicodedata.normalize('NFD', texto)
        if unicodedata.category(caracter) != 'Mn'
    )

# Respuestas locales de demostracion para las categorias indicadas por el Challenge.
PREGUNTAS_FRECUENTES_PEDIDOS = {
    "envio": "Realizamos envios nacionales. El tiempo estimado es de 3 a 5 dias habiles; la guia se comparte cuando el pedido sale del almacen.",
    "devolucion": "Puedes solicitar una devolucion dentro de los 30 dias naturales posteriores a la entrega. El producto debe estar sin uso y con su empaque original.",
    "pago": "Aceptamos tarjetas de credito y debito, y transferencia bancaria.",
    "factura": "Para solicitar factura, comparte tu RFC y numero de pedido con el equipo de atencion a clientes.",
}

def buscar_pregunta_frecuente(pregunta):
    pregunta_norm = quitar_acentos_pedidos(str(pregunta).lower())
    categorias = {
        "envio": ("envio", "paquete", "entrega", "guia", "rastreo"),
        "devolucion": ("devolucion", "devolver", "reembolso", "cambio"),
        "pago": ("pago", "pagar", "tarjeta", "transferencia", "metodo"),
        "factura": ("factura", "facturar", "rfc", "cfdi"),
    }
    for categoria, palabras_clave in categorias.items():
        if any(palabra in pregunta_norm for palabra in palabras_clave):
            return PREGUNTAS_FRECUENTES_PEDIDOS[categoria]
    return "No encontre una respuesta para esa pregunta en las preguntas frecuentes disponibles."

HERRAMIENTA_FAQ_PEDIDOS = {
    "type": "function",
    "name": "buscar_pregunta_frecuente",
    "description": "Busca respuestas locales sobre envios, devoluciones, formas de pago y facturas de la tienda. Usa la pregunta del cliente como argumento.",
    "parameters": {
        "type": "object",
        "properties": {
            "pregunta": {
                "type": "string",
                "description": "Pregunta del cliente sobre envios, devoluciones, formas de pago o facturas."
            }
        },
        "required": ["pregunta"]
    }
}

HERRAMIENTAS_PEDIDOS = [HERRAMIENTA_CONSULTAR_PEDIDO, HERRAMIENTA_FAQ_PEDIDOS]
FUNCIONES_DISPONIBLES_PEDIDOS = {
    "consultar_pedido": consultar_pedido,
    "buscar_pregunta_frecuente": buscar_pregunta_frecuente,
}

**3. Genera la respuesta con historial y function calling:** reenvía el historial a Llama junto con las dos herramientas; si el modelo pide ejecutar alguna, ejecútala de verdad y devuelve el resultado antes de generar la respuesta final.

In [ ]:
# Definir la funcion de generacion de respuesta con memoria y function calling

SYSTEM_PROMPT_PEDIDOS = (
    "Eres el asistente de atencion a clientes de una tienda en linea. "
    "Responde en espanol de forma clara y breve. "
    "Para informar el estado, fecha, productos o rastreo de un pedido, usa consultar_pedido "
    "y basa la respuesta exclusivamente en el resultado de esa herramienta. "
    "Nunca inventes numeros de pedido, estados, fechas, productos, transportistas ni guias. "
    "Si el cliente pregunta por un pedido pero no proporciona su numero, solicita ese numero y no lo supongas. "
    "Si el registro no existe, comunica que no se encontro en los registros disponibles; no inventes un resultado. "
    "Para preguntas sobre envios, devoluciones, formas de pago o facturas, usa buscar_pregunta_frecuente "
    "y contesta con la informacion devuelta. "
    "No afirmes informacion que no aparezca en la conversacion o en el resultado de una herramienta."
)

def generar_respuesta_pedido(numero, texto_usuario):
    conversation_id = obtener_conversacion_pedidos(numero)
    respuesta = client.responses.create(
        model="ollama/llama3.1:8b",
        input=texto_usuario,
        conversation=conversation_id,
        instructions=SYSTEM_PROMPT_PEDIDOS,
        tools=HERRAMIENTAS_PEDIDOS,
    )

    llamadas_funcion = [
        elemento for elemento in respuesta.output
        if elemento.type == "function_call"
    ]
    if llamadas_funcion:
        resultados = []
        for llamada in llamadas_funcion:
            funcion = FUNCIONES_DISPONIBLES_PEDIDOS[llamada.name]
            argumentos = json.loads(llamada.arguments)
            resultado = funcion(**argumentos)
            resultados.append({
                "type": "function_call_output",
                "call_id": llamada.call_id,
                "output": resultado,
            })

        # Los resultados se devuelven al mismo hilo; Llama redacta la respuesta final.
        respuesta = client.responses.create(
            model="ollama/llama3.1:8b",
            input=resultados,
            conversation=conversation_id,
            instructions=SYSTEM_PROMPT_PEDIDOS,
            tools=HERRAMIENTAS_PEDIDOS,
        )

    return respuesta.output_text

**4. Agrega las reglas fijas y protege el procesamiento por número:**

   * Si el usuario menciona un problema con un cargo (ej. "me cobraron dos veces", "cargo no reconocido", "fraude"), se responde con un mensaje fijo de prioridad, sin pasar por Llama.
   * Si el usuario pide hablar con una persona, se responde con un mensaje fijo de escalamiento.
   * Si el mensaje es solo un saludo ("hola", "buenas tardes"), se responde con un saludo fijo presentando al asistente.
   * Igual que en el Hands-On, usa un lock por número de teléfono para que dos mensajes seguidos del mismo usuario no disparen dos llamadas en paralelo sobre la misma conversación.
   * Registra en `errores_procesamiento` la causa de cada falla (Llama Stack, Ollama o WhatsApp) y avisa al usuario con un mensaje genérico, para que el bot nunca se quede mudo.

In [ ]:
# Un lock por numero de telefono, para serializar mensajes del mismo usuario

import threading

# Estructuras propias del Challenge; no comparten locks con el Hands-On.
locks_por_numero_pedidos = {}
_guard_locks_pedidos = threading.Lock()

def obtener_lock_pedidos(numero):
    clave = str(numero)
    with _guard_locks_pedidos:
        lock = locks_por_numero_pedidos.get(clave)
        if lock is None:
            lock = threading.Lock()
            locks_por_numero_pedidos[clave] = lock
        return lock

In [ ]:
# Definir funcion de procesamiento: reglas fijas de urgencia, escalamiento y saludo, o respuesta de Llama

from llama_stack_client import APITimeoutError, APIConnectionError, APIStatusError

# Reglas fijas del Challenge; se evalua cargo, escalamiento y saludo antes de llamar a Llama.
FRASES_PRIORIDAD_CARGO = [
    "me cobraron dos veces", "cargo no reconocido", "fraude",
    "cargo duplicado", "cobro duplicado", "me cobraron doble", "cargo doble",
]
FRASES_ESCALAMIENTO_PEDIDOS = [
    "hablar con una persona", "hablar con alguien", "asistente humano",
]
FRASES_SALUDO_PEDIDOS = [
    "hola", "buenas", "buenos dias", "buenas tardes", "buenas noches",
    "que tal", "buen dia",
]

MENSAJE_PRIORIDAD_CARGO = "Entiendo que reportas un problema con un cargo. Tu caso requiere atencion prioritaria; solicita apoyo a un asesor de la tienda."
MENSAJE_ESCALAMIENTO_PEDIDOS = "Claro, un miembro del equipo de atencion al cliente te ayudara."
MENSAJE_SALUDO_PEDIDOS = "Hola, soy el asistente de atencion a pedidos. \u00bfEn que puedo ayudarte?"
MENSAJE_ERROR_PEDIDOS = "Estamos teniendo un problema tecnico. Intenta de nuevo en un momento."

errores_procesamiento = []

def _sanitizar_error_pedido(error):
    texto_error = repr(error)
    for nombre in ("WHATSAPP_TOKEN", "VERIFY_TOKEN", "PHONE_ID", "WABA_ID", "NGROK_AUTHTOKEN"):
        secreto = globals().get(nombre)
        if isinstance(secreto, str) and secreto:
            texto_error = texto_error.replace(secreto, "[REDACTADO]")
    return texto_error

def registrar_error_procesamiento(numero, texto, causa, error, detalle):
    """Registra contexto interno sin devolver detalles tecnicos al usuario."""
    errores_procesamiento.append({
        "numero": str(numero),
        "texto": str(texto),
        "causa": causa,
        "detalle": detalle,
        "error": _sanitizar_error_pedido(error),
    })

def _causa_fallo_modelo(error):
    proceso_stack = globals().get("proceso_llama_stack")
    proceso_ollama = globals().get("proceso_ollama")
    stack_caido = proceso_stack is not None and proceso_stack.poll() is not None
    ollama_caido = proceso_ollama is not None and proceso_ollama.poll() is not None

    if stack_caido:
        return "llama_stack", "El proceso de Llama Stack no esta activo."
    if ollama_caido:
        return "ollama", "El proceso de Ollama no esta activo."
    if isinstance(error, APITimeoutError):
        return "llama_stack", "La solicitud al cliente de Llama Stack excedio el tiempo de espera; la causa interna puede ser el proveedor de inferencia."
    if isinstance(error, APIConnectionError):
        return "llama_stack", "No se pudo establecer conexion con el endpoint de Llama Stack."
    if isinstance(error, APIStatusError):
        return "llama_stack", f"Llama Stack devolvio HTTP {error.status_code}; si fue un error del proveedor Ollama no puede confirmarse en esta capa."
    return "desconocido", "Fallo no clasificado durante el procesamiento del mensaje."

def _texto_normalizado_reglas(texto):
    # La funcion de acentos pertenece al nucleo del Challenge (celda 56).
    normalizar = globals().get("quitar_acentos_pedidos", lambda valor: valor)
    return normalizar(str(texto).lower()).strip(" ,.!\u00a1\u00bf?")

def procesar_mensaje_pedido(numero, texto):
    """Procesa un mensaje bajo el lock del telefono y devuelve el texto de respuesta."""
    with obtener_lock_pedidos(numero):
        try:
            texto_reglas = _texto_normalizado_reglas(texto)
            if any(frase in texto_reglas for frase in FRASES_PRIORIDAD_CARGO):
                return MENSAJE_PRIORIDAD_CARGO
            if any(frase in texto_reglas for frase in FRASES_ESCALAMIENTO_PEDIDOS):
                return MENSAJE_ESCALAMIENTO_PEDIDOS
            if texto_reglas in FRASES_SALUDO_PEDIDOS:
                return MENSAJE_SALUDO_PEDIDOS

            return generar_respuesta_pedido(numero, texto)
        except Exception as error:
            causa, detalle = _causa_fallo_modelo(error)
            registrar_error_procesamiento(numero, texto, causa, error, detalle)
            return MENSAJE_ERROR_PEDIDOS

**5. Define los endpoints del webhook:** usa `BackgroundTasks` para responder de inmediato a Meta, delegando el procesamiento real a la función anterior. Ignora los mensajes que Meta reenvíe con el mismo `wamid`, para no responderlos dos veces.

In [ ]:
# Configurar el servidor del webhook (Challenge)

from fastapi import FastAPI, Request, BackgroundTasks
from fastapi.responses import PlainTextResponse, Response
import hmac
import json
import requests
import threading

# Aplicacion y registros propios del agente de pedidos; no se comparte el app del taller.
app_pedidos = FastAPI()
mensajes_recibidos_pedidos = []
respuestas_enviadas_pedidos = []

# Los IDs quedan marcados al aceptarse de forma atomica para programar su tarea.
wamids_aceptados_pedidos = set()
_guard_wamids_pedidos = threading.Lock()

In [ ]:
# Definir funcion de normalizacion de numero para el Challenge

def normalizar_numero_mx_pedidos(numero):
    if numero.startswith("521") and len(numero) == 13:
        return "52" + numero[3:]
    return numero

In [ ]:
# Definir endpoint de verificacion de credenciales

@app_pedidos.get("/webhook")
async def verificar_webhook_pedidos(request: Request):
    params = request.query_params
    modo = params.get("hub.mode")
    token_recibido = params.get("hub.verify_token")
    challenge = params.get("hub.challenge")

    if (
        modo == "subscribe"
        and isinstance(VERIFY_TOKEN, str)
        and isinstance(token_recibido, str)
        and hmac.compare_digest(token_recibido, VERIFY_TOKEN)
        and challenge is not None
        and challenge != ""
    ):
        return PlainTextResponse(content=str(challenge), status_code=200)

    # Respuesta vacia: no revela el token esperado ni el token recibido.
    return Response(status_code=403)


In [ ]:
# Definir endpoint de recepcion con validacion de firma oficial de Meta

import hashlib
import re

@app_pedidos.post("/webhook")
async def recibir_mensaje_pedidos(request: Request, background_tasks: BackgroundTasks):
    # Validar el formato y X-Hub-Signature-256 sobre bytes originales antes de cualquier efecto lateral.
    raw_body = await request.body()
    firma_recibida = request.headers.get("X-Hub-Signature-256")
    if not isinstance(META_APP_SECRET, str) or not META_APP_SECRET or not isinstance(firma_recibida, str):
        return Response(status_code=401)
    if re.fullmatch(r"sha256=[0-9a-fA-F]{64}", firma_recibida) is None:
        return Response(status_code=401)
    firma_esperada = "sha256=" + hmac.new(
        META_APP_SECRET.encode("utf-8"), raw_body, hashlib.sha256
    ).hexdigest()
    if not hmac.compare_digest(firma_recibida, firma_esperada):
        return Response(status_code=401)

    try:
        body = json.loads(raw_body)
    except (json.JSONDecodeError, UnicodeDecodeError):
        return {"status": "ignored"}

    if not isinstance(body, dict):
        return {"status": "ignored"}

    aceptados = 0
    entries = body.get("entry")
    if not isinstance(entries, list):
        return {"status": "ok", "scheduled": aceptados}

    for entry in entries:
        if not isinstance(entry, dict):
            continue
        changes = entry.get("changes")
        if not isinstance(changes, list):
            continue

        for change in changes:
            value = change.get("value") if isinstance(change, dict) else None
            if not isinstance(value, dict):
                continue
            messages = value.get("messages")
            if not isinstance(messages, list):
                continue  # estados y notificaciones sin mensajes se ignoran

            for message in messages:
                if not isinstance(message, dict) or message.get("type") != "text":
                    continue
                wamid = message.get("id")
                remitente = message.get("from")
                text_data = message.get("text")
                texto = text_data.get("body") if isinstance(text_data, dict) else None
                if not all(isinstance(valor, str) and valor.strip() for valor in (wamid, remitente, texto)):
                    continue

                numero = normalizar_numero_mx_pedidos(remitente)
                # Check, reserva del ID y registro de la tarea ocurren bajo el mismo lock.
                with _guard_wamids_pedidos:
                    if wamid in wamids_aceptados_pedidos:
                        continue
                    wamids_aceptados_pedidos.add(wamid)
                    try:
                        background_tasks.add_task(
                            procesar_y_enviar_respuesta_pedido, numero, texto
                        )
                    except Exception:
                        wamids_aceptados_pedidos.discard(wamid)
                        raise

                mensajes_recibidos_pedidos.append({
                    "wamid": wamid,
                    "numero": numero,
                    "texto": texto,
                })
                aceptados += 1

    return {"status": "ok", "scheduled": aceptados}


In [ ]:
# Definir funcion de envio de respuesta mediante WhatsApp

def enviar_respuesta_pedido(numero, texto):
    url = f"https://graph.facebook.com/v25.0/{PHONE_ID}/messages"
    headers = {
        "Authorization": f"Bearer {WHATSAPP_TOKEN}",
        "Content-Type": "application/json",
    }
    payload = {
        "messaging_product": "whatsapp",
        "to": numero,
        "type": "text",
        "text": {"body": texto},
    }

    try:
        if not WHATSAPP_TOKEN or not PHONE_ID:
            raise RuntimeError("Faltan credenciales de WhatsApp para enviar el mensaje.")
        respuesta = requests.post(url, headers=headers, json=payload, timeout=15)
        respuesta.raise_for_status()
        respuestas_enviadas_pedidos.append({
            "numero": numero,
            "status": respuesta.status_code,
        })
        return True
    except requests.exceptions.Timeout as error:
        registrar_error_procesamiento(
            numero, texto, "whatsapp", type(error).__name__,
            "Tiempo de espera agotado al enviar el mensaje por WhatsApp Graph API.",
        )
    except requests.exceptions.HTTPError as error:
        codigo = getattr(getattr(error, "response", None), "status_code", None)
        registrar_error_procesamiento(
            numero, texto, "whatsapp", type(error).__name__,
            f"WhatsApp Graph API devolvio una respuesta HTTP no exitosa (status={codigo}).",
        )
    except requests.exceptions.RequestException as error:
        registrar_error_procesamiento(
            numero, texto, "whatsapp", type(error).__name__,
            "Fallo de red al enviar el mensaje por WhatsApp Graph API.",
        )
    except Exception as error:
        registrar_error_procesamiento(
            numero, texto, "whatsapp", type(error).__name__,
            "No se pudo preparar o enviar el mensaje por WhatsApp Graph API.",
        )
    return False


def procesar_y_enviar_respuesta_pedido(numero, texto):
    respuesta = procesar_mensaje_pedido(numero, texto)
    return enviar_respuesta_pedido(numero, respuesta)

**6. Expón el servidor con ngrok y prueba el flujo completo:**

   * Manda un mensaje preguntando por el estado de un pedido y confirma que la respuesta viene de `consultar_pedido` (dato real), no de una invención de Llama.
   * Manda una pregunta frecuente y confirma que dispara `buscar_pregunta_frecuente`.
   * Manda un mensaje sobre un cargo duplicado y confirma que dispara la regla de urgencia, sin pasar por Llama.
   * Manda un mensaje pidiendo hablar con una persona y confirma que dispara la regla de escalamiento.

In [ ]:
# Instalar e iniciar el servidor Uvicorn y el tunel publico con ngrok (Colab)

import importlib.util
import subprocess
import sys
import threading
import time
import urllib.error
import urllib.request

paquetes_exposicion = {"pyngrok": "pyngrok", "uvicorn": "uvicorn"}
paquetes_faltantes = [
    paquete for modulo, paquete in paquetes_exposicion.items()
    if importlib.util.find_spec(modulo) is None
]
if paquetes_faltantes:
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "--quiet", *paquetes_faltantes],
        check=True,
    )

from google.colab import userdata
from pyngrok import ngrok
import uvicorn

PUERTO_WEBHOOK_PEDIDOS = 8000
NGROK_AUTHTOKEN = userdata.get("NGROK_AUTHTOKEN")
if not NGROK_AUTHTOKEN:
    raise RuntimeError("Agrega NGROK_AUTHTOKEN a Colab Secrets antes de ejecutar esta celda.")

# Configurar el token desde Colab Secrets sin imprimirlo.
ngrok.set_auth_token(NGROK_AUTHTOKEN)

# Reutilizar el servidor creado por esta celda al volver a ejecutarla.
if (
    "servidor_uvicorn_pedidos" not in globals()
    or "hilo_uvicorn_pedidos" not in globals()
    or not hilo_uvicorn_pedidos.is_alive()
):
    servidor_uvicorn_pedidos = uvicorn.Server(
        uvicorn.Config(
            app_pedidos,
            host="0.0.0.0",
            port=PUERTO_WEBHOOK_PEDIDOS,
            log_level="warning",
        )
    )
    hilo_uvicorn_pedidos = threading.Thread(
        target=servidor_uvicorn_pedidos.run,
        name="uvicorn-app-pedidos",
        daemon=True,
    )
    hilo_uvicorn_pedidos.start()

# Esperar a que el puerto publicado responda antes de crear/reutilizar el tunel.
url_local_webhook_pedidos = f"http://127.0.0.1:{PUERTO_WEBHOOK_PEDIDOS}/webhook"
limite_inicio_uvicorn = time.time() + 30
while time.time() < limite_inicio_uvicorn:
    if not hilo_uvicorn_pedidos.is_alive():
        raise RuntimeError("Uvicorn termino antes de iniciar el webhook de pedidos.")
    try:
        urllib.request.urlopen(url_local_webhook_pedidos, timeout=2)
        break
    except urllib.error.HTTPError:
        # GET /webhook sin parametros puede responder 403; eso confirma el servidor.
        break
    except urllib.error.URLError:
        time.sleep(0.5)
else:
    raise RuntimeError("Uvicorn no respondio en el puerto 8000 dentro de 30 segundos.")

# Si la celda se repite, reutiliza el tunel activo en vez de abrir otro.
tuneles_existentes_pedidos = ngrok.get_tunnels()
tunel_previo_pedidos = globals().get("tunel_ngrok_pedidos")
url_previa_pedidos = getattr(tunel_previo_pedidos, "public_url", None)
tunel_ngrok_pedidos = next(
    (
        tunel for tunel in tuneles_existentes_pedidos
        if (
            getattr(tunel, "public_url", None) == url_previa_pedidos
            or str(getattr(tunel, "config", {}).get("addr", "")).endswith(
                f":{PUERTO_WEBHOOK_PEDIDOS}"
            )
        )
    ),
    None,
)
if tunel_ngrok_pedidos is None:
    tunel_ngrok_pedidos = ngrok.connect(addr=PUERTO_WEBHOOK_PEDIDOS, proto="http")

URL_PUBLICA_PEDIDOS = tunel_ngrok_pedidos.public_url
URL_WEBHOOK_META_PEDIDOS = URL_PUBLICA_PEDIDOS.rstrip("/") + "/webhook"
print("Servidor Uvicorn: app_pedidos en el puerto", PUERTO_WEBHOOK_PEDIDOS)
print("URL publica de ngrok:", URL_PUBLICA_PEDIDOS)
print("URL final del webhook para configurar en Meta:", URL_WEBHOOK_META_PEDIDOS)


In [ ]:
# Escenario 1: Mandar mensajes al bot por tema de pedidos y preguntas frecuentes
# GUIA DE PRUEBA: son mensajes/resultados ESPERADOS, no resultados observados.
# Enviar manualmente desde WhatsApp tras configurar el webhook en Meta.

from IPython.display import display, Markdown

display(Markdown("""
### Escenario 1 - Pedido existente y continuidad conversacional

1. **Mensaje:** ¿Cual es el estado de mi pedido PED-1002?
   - **Esperado:** interviene Llama y solicita `consultar_pedido` con `numero_pedido="PED-1002"`.
   - **Resultado funcional esperado:** respuesta final basada en el registro local PED-1002 (estado Enviado y datos almacenados). Esto no se considera observado hasta probarlo en WhatsApp.
2. **Seguimiento en el mismo chat:** ¿Cual es la guia de ese pedido?
   - **Esperado:** se reutiliza la conversacion Llama Stack del mismo telefono y el contexto PED-1002.
   - **Herramienta esperada:** `consultar_pedido` para responder con los datos locales de ese pedido.

### Escenario 2 - Pregunta frecuente

- **Mensaje:** ¿Cuanto tarda el envio?
- **Esperado:** interviene Llama y solicita `buscar_pregunta_frecuente` con la pregunta del cliente.
- **Resultado funcional esperado:** respuesta derivada de la FAQ local de envios.

**Estado:** guion/resultado esperado solamente; no se enviaron mensajes ni se observaron respuestas.
"""))


In [ ]:
# Escenario 2: Mandar mensajes al bot por tema de cargos
# GUIA DE PRUEBA: comportamiento esperado; todavia no es evidencia real.

from IPython.display import display, Markdown

display(Markdown("""
### Escenario 3 - Cargo duplicado / prioridad fija

- **Mensaje:** Me cobraron dos veces el pedido.
- **Esperado:** responde la regla fija de prioridad del Challenge.
- **Llama/function calling:** no debe intervenir Llama ni ejecutarse una herramienta.
- **Resultado funcional esperado:** mensaje fijo de prioridad definido en la celda 61.

### Escenario 4 - Solicitud de una persona

- **Mensaje:** Quiero hablar con una persona, por favor.
- **Esperado:** responde la regla fija de escalamiento.
- **Llama/function calling:** no debe intervenir Llama ni ejecutarse una herramienta.
- **Resultado funcional esperado:** mensaje fijo de escalamiento definido en la celda 61.

**Estado:** guion/resultado esperado solamente; no se enviaron mensajes ni se observaron respuestas.
"""))


In [ ]:
# Escenario 3: Mandar mensajes por un pedido sin numero y por un numero inexistente
# GUIA DE PRUEBA: comportamiento esperado; no simula respuestas del modelo.

from IPython.display import display, Markdown

display(Markdown("""
### Escenario 5 - Consulta de pedido sin numero

- **Mensaje:** ¿Donde esta mi pedido?
- **Esperado:** Llama solicita el numero al usuario.
- **Function calling:** no debe ejecutarse `consultar_pedido` con un identificador inventado.
- **Resultado funcional esperado:** se pide el numero de pedido para continuar.

### Escenario 6 - Numero de pedido inexistente

- **Mensaje:** ¿Cual es el estado de mi pedido PED-9999?
- **Esperado:** interviene Llama y solicita `consultar_pedido` con `numero_pedido="PED-9999"`.
- **Resultado funcional esperado:** la herramienta informa que el pedido no existe y Llama lo comunica sin inventar estado ni datos.

**Estado:** guion/resultado esperado solamente; no se enviaron mensajes ni se observaron respuestas.
"""))


In [ ]:
# Visualizar resultados realmente almacenados durante la ejecucion del Challenge

from pprint import pprint

print("Mensajes recibidos (wamid, numero y texto):")
pprint(mensajes_recibidos_pedidos)
print("\nRespuestas enviadas (numero y estado HTTP; sin tokens):")
pprint(respuestas_enviadas_pedidos)
print("\nwamids aceptados para procesamiento:")
pprint(sorted(wamids_aceptados_pedidos))
print("\nErrores registrados (sin encabezados ni tokens de autenticacion):")
pprint(errores_procesamiento)


In [ ]:
# Verificar almacenamiento de las conversaciones del Challenge en Llama Stack

from pprint import pprint

print("Mapa Challenge: telefono -> conversation_id")
pprint(conversaciones_por_numero_pedidos)

for numero, conversation_id in conversaciones_por_numero_pedidos.items():
    print(f"\nElementos almacenados en Llama Stack para {numero} (conversation_id={conversation_id}):")
    try:
        elementos = client.conversations.items.list(conversation_id=conversation_id)
        # Mostrar lo devuelto por la API al ejecutar esta celda.
        pprint(elementos.model_dump() if hasattr(elementos, "model_dump") else elementos)
    except Exception as error:
        # No imprimir mensajes de excepcion que pudieran incluir detalles sensibles.
        print("No se pudieron consultar los elementos:", type(error).__name__)


In [ ]:
# Finalizar el tunel ngrok y limpiar estructuras en memoria del Challenge

# Cerrar solamente el tunel publico usado por app_pedidos.
try:
    from pyngrok import ngrok
    url_tunel = globals().get("URL_PUBLICA_PEDIDOS")
    if url_tunel:
        ngrok.disconnect(url_tunel)
        print("Tunel ngrok del Challenge cerrado.")
    else:
        print("No hay un tunel ngrok del Challenge que cerrar.")
except Exception as error:
    print("No se pudo confirmar el cierre de ngrok:", type(error).__name__)

# Solicitar cierre ordenado de Uvicorn si esta celda lo inicio.
servidor = globals().get("servidor_uvicorn_pedidos")
hilo_servidor = globals().get("hilo_uvicorn_pedidos")
if servidor is not None and hilo_servidor is not None and hilo_servidor.is_alive():
    servidor.should_exit = True
    hilo_servidor.join(timeout=10)
    print("Solicitud de cierre enviada a Uvicorn.")

# Limpiar solamente las estructuras de memoria propias del Challenge.
conversaciones_por_numero_pedidos.clear()
with _guard_locks_pedidos:
    locks_por_numero_pedidos.clear()
with _guard_wamids_pedidos:
    wamids_aceptados_pedidos.clear()
mensajes_recibidos_pedidos.clear()
respuestas_enviadas_pedidos.clear()
errores_procesamiento.clear()
print("Estructuras en memoria del Challenge limpiadas.")
